# Python `unittest`

## Einführung: Warum testen?

Tests sind automatisierte Prüfungen, die sicherstellen, dass Code **wie erwartet** funktioniert – heute und auch nach Änderungen in der Zukunft.

### Warum lohnt sich das?
- **Fehler zeitnah finden**
- **Dokumentation durch Beispiele** (Tests zeigen, wie etwas benutzt werden soll)
- **Refactoring mit Vertrauen**

### Begriffe
- **Unit Test**: testet eine "kleine" Einheit (z.B. eine Funktion) isoliert, das heißt ohne potenziell ungetestete Abhängigkeiten
- **Integration Test**: testet das Zusammenspiel mehrerer Komponenten
- **End-to-End Test**: testet den kompletten Ablauf aus Nutzerperspektive

### Ablauf eines Test-Skripts
1. **Arrange**: Setup der Testdaten / Umgebung
2. **Act**: Code ausführen
3. **Assert**: Ergebnis überprüfen

## Was ist `unittest`?

`unittest` ist Pythons eingebautes Test-Framework (ähnlich JUnit). Es bietet:
- Testklassen mit **Testmethoden**
- Satz komfortabler **Assertions** (z.B. `assertEqual`, `assertTrue`, `assertRaises` …)
- **Fixtures** (`setUp`, `tearDown`, …)
- Test Runner & **Test Discovery**
- **Mocking** über `unittest.mock`
    - Durch Mocking werden Abhängigkeiten durch eine konfigurierbare Attrappe ersetzt 

## Ein erster Test

Ein `unittest`-Test besteht typischerweise aus:
- einer Klasse, die von `unittest.TestCase` erbt
- Methoden, deren Name mit `test_` beginnt

Wir starten mit einer Mini-Funktion und testen sie.

In [1]:
def add(a, b):
    return a + b

In [2]:
import unittest

class TestAdd(unittest.TestCase):
    def test_add_two_numbers(self):
        # Arrange
        a, b = 2, 3

        # Act
        result = add(a, b)

        # Assert
        self.assertEqual(result, 5)

### Tests im Notebook ausführen

* In einem realen Projekt wird die Test Discovery genutzt
 * `python -m unittest`
* Standalone oder im Notebook kann der Test Runner direkt gestartet werden:
    * unittest.main()` versucht Notebook-Argumente zu parsen → deshalb nutzen wir `argv=['']`.
    * exit=False` verhindert, dass das Notebook stoppt.

In [3]:
unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.005s

OK


## Assertions

* Assertions prüfen in einer simplen Anweisung eine Erwartung

    - `assertEqual(a, b)` / `assertNotEqual(a, b)`
    - `assertTrue(x)` / `assertFalse(x)`
    - `assertIs(a, b)` / `assertIsNone(x)`
    - `assertIn(item, container)`
    - `assertAlmostEqual(a, b, places=...)` für Fließkomma
    - `assertRaises(ExceptionType)` für Exceptions

In [4]:
class TestAssertions(unittest.TestCase):
    def test_examples(self):
        self.assertTrue(1 < 2)
        self.assertIn("py", "python")
        self.assertAlmostEqual(0.1 + 0.2, 0.3, places=7)  # floating point!

    def test_exception(self):
        with self.assertRaises(ZeroDivisionError):
            _ = 1 / 0

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


## Test-Discovery

In echten Projekten liegen Tests meistens in einem `tests/`-Ordner.

Ein gängiges Layout:

```
my_project/
  src/my_project/
    __init__.py
    calc.py
  tests/
    test_calc.py
```
In diesem Standard-Layout werden Tests automatisch gefunden:

```bash
python -m unittest discover -s tests -p "test_*.py" -v
```

**Merke:**
- Dateien, Klassen und Methoden sollten sinnvoll benannt sein.
- `test_*.py` ist ein verbreitetes Pattern.

## Fixtures: `setUp` und `tearDown`

Wenn mehrere Tests dasselbe Setup brauchen (z.B. Testdaten, temporäre Dateien, Clients …),
werden Fixtures verwendet:

- `setUp()` läuft **vor jedem** Test
- `tearDown()` läuft **nach jedem** Test
- `setUpClass()` / `tearDownClass()` laufen **einmal pro Klasse**

In [5]:
class ShoppingCart:
    def __init__(self):
        self.items = []
    def add(self, name, price):
        self.items.append((name, price))
    def total(self):
        return sum(price for _, price in self.items)

class TestShoppingCart(unittest.TestCase):
    def setUp(self):
        self.cart = ShoppingCart()
        self.cart.add("apple", 1.20)

    def tearDown(self):
        # hier würdest du z.B. temporäre Ressourcen wieder freigeben
        self.cart = None

    def test_total_with_one_item(self):
        self.assertAlmostEqual(self.cart.total(), 1.20, places=2)

    def test_total_with_two_items(self):
        self.cart.add("banana", 0.80)
        self.assertAlmostEqual(self.cart.total(), 2.00, places=2)

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok
test_total_with_one_item (__main__.TestShoppingCart.test_total_with_one_item) ... ok
test_total_with_two_items (__main__.TestShoppingCart.test_total_with_two_items) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.017s

OK


## Subtests

Soll ein Test eine Kombination von Eingangsdaten für eine Testsequenz nutzen wird `subTest` genutzt:
- **Ein** Test mit mehreren Unterfällen
- Fehlern werden dem jeweiligen Subtest zugeordnet

In [6]:
def bmi(height_cm, weight_kg):
    h = height_cm / 100
    return weight_kg / (h*h)

class TestBMI(unittest.TestCase):
    def test_bmi_multiple_cases(self):
        cases = [
            (180, 81, 25.0),
            (170, 68, 23.5294117647),
            (160, 50, 19.53125),
        ]
        for height, weight, expected in cases:
            with self.subTest(height=height, weight=weight):
                self.assertAlmostEqual(bmi(height, weight), expected, places=6)

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok
test_bmi_multiple_cases (__main__.TestBMI.test_bmi_multiple_cases) ... ok
test_total_with_one_item (__main__.TestShoppingCart.test_total_with_one_item) ... ok
test_total_with_two_items (__main__.TestShoppingCart.test_total_with_two_items) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.021s

OK


## Skips & erwartete Fehler

Tests können übersprungen werden:
- Feature ist noch nicht fertig implementiert
- ein Test ist plattformabhängig und muss deshalb temporär für einen TEstlauf deaktiviert werden

Tests können auch so formuliert werden, dass ein **Fehlschlag erwartet** wird
- Beispiel: Es wird erwartet, dass ein Feature noch nicht korrekt implementiert ist
    - Hinweis: Ein eher diskussionswürdiges Szenarium...

In [7]:
import sys

class TestSkips(unittest.TestCase):
    @unittest.skip("Beispiel: Test noch nicht relevant")
    def test_skipped(self):
        self.fail("Wird nie ausgeführt")

    @unittest.skipIf(sys.platform.startswith("win"), "Beispiel: funktioniert nicht unter Windows")
    def test_skip_if(self):
        self.assertTrue(True)

    @unittest.expectedFailure
    def test_expected_failure(self):
        self.assertEqual(1 + 1, 3)

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok
test_bmi_multiple_cases (__main__.TestBMI.test_bmi_multiple_cases) ... ok
test_total_with_one_item (__main__.TestShoppingCart.test_total_with_one_item) ... ok
test_total_with_two_items (__main__.TestShoppingCart.test_total_with_two_items) ... 

ok
test_expected_failure (__main__.TestSkips.test_expected_failure) ... expected failure
skipped 'Beispiel: funktioniert nicht unter Windows'
test_skipped (__main__.TestSkips.test_skipped) ... skipped 'Beispiel: Test noch nicht relevant'

----------------------------------------------------------------------
Ran 7 tests in 0.033s

OK (skipped=2, expected failures=1)


## Tests auf geworfene Exceptions
- Dazu dient die spezielle Assertion `assertRaises`
    - In Verbindung mit dem Context Manager ist auch der Zugriff auf die Fehlermeldungmöglich


In [8]:
def parse_age(value: str) -> int:
    if not value.isdigit():
        raise ValueError(f"Invalid age: {value!r}")
    return int(value)

class TestParseAge(unittest.TestCase):
    def test_parse_ok(self):
        self.assertEqual(parse_age("42"), 42)

    def test_parse_invalid(self):
        with self.assertRaises(ValueError) as ctx:
            parse_age("forty-two")
        self.assertIn("Invalid age", str(ctx.exception))

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok
test_bmi_multiple_cases (__main__.TestBMI.test_bmi_multiple_cases) ... ok
test_parse_invalid (__main__.TestParseAge.test_parse_invalid) ... ok
test_parse_ok (__main__.TestParseAge.test_parse_ok) ... ok
test_total_with_one_item (__main__.TestShoppingCart.test_total_with_one_item) ... ok
test_total_with_two_items (__main__.TestShoppingCart.test_total_with_two_items) ... ok
test_expected_failure (__main__.TestSkips.test_expected_failure) ... expected failure
skipped 'Beispiel: funktioniert nicht unter Windows'
test_skipped (__main__.TestSkips.test_skipped) ... skipped 'Beispiel: Test noch nicht relevant'

----------------------------------------------------------------------
Ran 9 tests in 0.030s

OK (skipped=2, expected failures=1)


## Eine Einführung in `unittest.mock`

Reale Applikationen nutzen fast immer irgendwelche Backend-Systeme / Ressourcen:
- HTTP Requests
- Datenbanken
- Zeit, Zufall, Dateisystem
- andere Services

Mit Hilfe von **Mocks** (Test-Doubles, Attrappen-Objekte) können diese Abhängigkeiten ersetzt werden
- Damit kann beispielsweise ein Datenbankzugriff ohne das Aufsetzen einer physikalischen Datenbank getestet werden

Bekannte Module:
- `Mock`
- `MagicMock`


In [9]:
from unittest.mock import Mock, patch

def get_exchange_rate(api_client, base: str, target: str) -> float:
    # api_client.fetch(...) soll z.B. ein dict liefern: {"rate": 0.92}
    data = api_client.fetch(base=base, target=target)
    return float(data["rate"])

class TestMockBasics(unittest.TestCase):
    def test_mock_client(self):
        fake_client = Mock()
        fake_client.fetch.return_value = {"rate": 0.92}

        rate = get_exchange_rate(fake_client, "USD", "EUR")

        self.assertEqual(rate, 0.92)
        fake_client.fetch.assert_called_once_with(base="USD", target="EUR")

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok
test_bmi_multiple_cases (__main__.TestBMI.test_bmi_multiple_cases) ... ok
test_mock_client (__main__.TestMockBasics.test_mock_client) ... ok
test_parse_invalid (__main__.TestParseAge.test_parse_invalid) ... ok
test_parse_ok (__main__.TestParseAge.test_parse_ok) ... ok
test_total_with_one_item (__main__.TestShoppingCart.test_total_with_one_item) ... ok
test_total_with_two_items (__main__.TestShoppingCart.test_total_with_two_items) ... ok
test_expected_failure (__main__.TestSkips.test_expected_failure) ... expected failure
skipped 'Beispiel: funktioniert nicht unter Windows'
test_skipped (__main__.TestSkips.test_skipped) ... skipped 'Beispiel: Test noch nicht relevant'

----------------------------------------------------------------------
Ran 10 tests in 0.047s

OK (skipped=2, expected failures=1)


### `patch()` – eine Funktion/Variable zur Laufzeit ersetzen

Typischer Use-Case: Eine Funktion ruft intern `time.time()` oder `requests.get(url)` auF
- Was für eine Assertion definieren wir für die aktuelle Zeit auf?
- Die `url`des GET-Requests muss einen realen Endpunkt aufrufen

Ein `patch()` ersetzt den Aufruf einer Methode mit einer wohldefinierten Rückgabe

In [10]:
import time

def is_cache_expired(created_at_ts: float, ttl_seconds: int) -> bool:
    return (time.time() - created_at_ts) > ttl_seconds

class TestPatch(unittest.TestCase):
    def test_cache_not_expired(self):
        created = 1_000.0
        ttl = 10

        with patch("time.time", return_value=1_005.0):
            self.assertFalse(is_cache_expired(created, ttl))

    def test_cache_expired(self):
        created = 1_000.0
        ttl = 10

        with patch("time.time", return_value=1_020.0):
            self.assertTrue(is_cache_expired(created, ttl))

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok
test_bmi_multiple_cases (__main__.TestBMI.test_bmi_multiple_cases) ... ok
test_mock_client (__main__.TestMockBasics.test_mock_client) ... ok
test_parse_invalid (__main__.TestParseAge.test_parse_invalid) ... ok
test_parse_ok (__main__.TestParseAge.test_parse_ok) ... ok
test_cache_expired (__main__.TestPatch.test_cache_expired) ... ok
test_cache_not_expired (__main__.TestPatch.test_cache_not_expired) ... ok
test_total_with_one_item (__main__.TestShoppingCart.test_total_with_one_item) ... ok
test_total_with_two_items (__main__.TestShoppingCart.test_total_with_two_items) ... ok
test_expected_failure (__main__.TestSkips.test_expected_failure) ... expected failure
skipped 'Beispiel: funktioniert nicht unter Windows'
test_skipped (__main__.TestSkips.test_skipped) ... skipped 'Beispiel: Test noch nicht re

## Testqualität

**Does**
- **deterministisch** (kein Zufall, keine echte Zeit, keine echte Netzwerkabhängigkeit)
- **schnell** (Unit Tests sollten in Sekunden laufen)
- **präzise** (ein klarer Grund, warum ein Test scheitert)
- **wartbar** (wenig Duplikate, sprechende Namen)

**Dont's**
- Tests, die aufeinander aufbauen (Reihenfolgeabhängigkeit)
- zu viel Logik im Test selbst
- übermäßiges Mocking (Test spiegelt nur die Implementierung wider statt Verhalten)

## Ideen für Übungsaufgaben

### Aufgabe A
Tests für:
- `normalize_name("  aLiCe  ") -> "Alice"`
- Leere/Whitespace-Strings sollen `ValueError` werfen

### Aufgabe B
Tests für `safe_div(a, b)`:
- normale Division
- Division durch 0 soll `ZeroDivisionError` werfen

In [11]:
def normalize_name(s: str) -> str:
    cleaned = s.strip()
    if not cleaned:
        raise ValueError("name must not be empty")
    return cleaned.capitalize()

def safe_div(a, b):
    return a / b

In [12]:
class TestExercises(unittest.TestCase):
    # Aufgabe A
    def test_normalize_name_ok(self):
        self.assertEqual(normalize_name("  aLiCe  "), "Alice")

    def test_normalize_name_empty(self):
        with self.assertRaises(ValueError):
            normalize_name("   ")

    # Aufgabe B
    def test_safe_div_ok(self):
        self.assertEqual(safe_div(10, 2), 5)

    def test_safe_div_zero(self):
        with self.assertRaises(ZeroDivisionError):
            safe_div(10, 0)

unittest.main(argv=[''], exit=False, verbosity=2)

test_add_two_numbers (__main__.TestAdd.test_add_two_numbers) ... ok
test_examples (__main__.TestAssertions.test_examples) ... ok
test_exception (__main__.TestAssertions.test_exception) ... ok
test_bmi_multiple_cases (__main__.TestBMI.test_bmi_multiple_cases) ... ok
test_normalize_name_empty (__main__.TestExercises.test_normalize_name_empty) ... ok
test_normalize_name_ok (__main__.TestExercises.test_normalize_name_ok) ... ok
test_safe_div_ok (__main__.TestExercises.test_safe_div_ok) ... ok
test_safe_div_zero (__main__.TestExercises.test_safe_div_zero) ... ok
test_mock_client (__main__.TestMockBasics.test_mock_client) ... ok
test_parse_invalid (__main__.TestParseAge.test_parse_invalid) ... ok
test_parse_ok (__main__.TestParseAge.test_parse_ok) ... ok
test_cache_expired (__main__.TestPatch.test_cache_expired) ... ok
test_cache_not_expired (__main__.TestPatch.test_cache_not_expired) ... ok
test_total_with_one_item (__main__.TestShoppingCart.test_total_with_one_item) ... ok
test_total_with_